## The cities file: five delivery cities in one dictionary

## Virtual environment and pip install requests

## requests.get with params, timeout and raise_for_status

In [1]:
import requests
from cities import cities
api = "https://archive-api.open-meteo.com"
api += "/v1/archive"
lat, lon = cities["Mumbai"]
ask = {"daily": "rain_sum"}
ask["latitude"] = lat
ask["longitude"] = lon
ask["start_date"] = "2025-08-19"
ask["end_date"] = "2025-08-19"
ask["timezone"] = "Asia/Kolkata"
r = requests.get(api, params=ask, timeout=30)
r.raise_for_status()
print(r.status_code)
print(r.headers["Content-Type"])
print(r.json()["daily"])

200
application/json; charset=utf-8
{'time': ['2025-08-19'], 'rain_sum': [136.5]}


## A wrong date: reading a 400 status and the reason

In [2]:
bad = dict(ask, end_date="2025-13-01")
r = requests.get(api, params=bad, timeout=30)
print(r.status_code)
print(r.json()["reason"])
print(r.json().get("daily"))

400
Invalid date
None


## json_normalize: rain for five cities in one flat table

In [3]:
import pandas as pd
rows = []
for name, (lat, lon) in cities.items():
    ask["latitude"] = lat
    ask["longitude"] = lon
    r = requests.get(api, params=ask, timeout=30)
    r.raise_for_status()
    one = r.json()
    one["city"] = name
    rows.append(one)
df = pd.json_normalize(rows)
df["rain"] = df["daily.rain_sum"].str[0]
print(df[["city", "rain"]])

        city   rain
0     Mumbai  136.5
1      Delhi    5.4
2  Bengaluru    0.3
3    Chennai    0.5
4    Kolkata   10.0
